# lean-agent — showcase

What the repo does, built up explicitly so nothing is hidden. The question it exists to answer: **does giving the agent well-structured definitions make it prove Lean theorems more effectively?**

Setup (once): `uv sync`, put `OPENAI_API_KEY` in `.env`, and install Lean:

```sh
curl -sSf https://elan.lean-lang.org/elan-init.sh | sh -s -- -y
~/.elan/bin/elan default leanprover/lean4:v4.29.1
```

The next cell makes the project importable and puts `~/.elan/bin` on `PATH` (Jupyter kernels usually don't inherit your shell `PATH` or start from the project root). The first Lean call builds the REPL (one-time).

In [1]:
import os
import sys
from pathlib import Path

# Make the top-level `benchmarks/` package importable (the kernel's cwd is often notebooks/).
for _root in (Path.cwd(), *Path.cwd().parents):
    if (_root / "benchmarks" / "__init__.py").exists():
        sys.path.insert(0, str(_root))
        break

# Put elan's Lean toolchain (where `lake` / `lean` live) on PATH before the first Lean call.
elan_bin = os.path.expanduser("~/.elan/bin")
if os.path.isdir(elan_bin) and elan_bin not in os.environ["PATH"].split(os.pathsep):
    os.environ["PATH"] = elan_bin + os.pathsep + os.environ["PATH"]

from lean_agent import Lean, lean_config, Problem, solve, build_model, get_settings

s = get_settings()
print("model:", s.model_id, "| lean:", s.lean_version, "| key set:", bool(s.api_key))

model: gpt-5.4-mini | lean: v4.29.1 | key set: True


## 1. Talk to Lean directly: define a concept, then prove against it

This is the core mechanism. A **preamble** (some Lean code — imports, definitions, lemmas) is run **once** to build a persistent Lean environment. Later proofs run *inside* that environment, so the definitions are in scope. Here we use plain core Lean, no Mathlib.

In [2]:
# `definitions` is a string of LEAN code (not Python): a definition + a helper lemma.
definitions = (
    "def IsEven (n : Nat) : Prop := ∃ k, n = 2 * k\n"
    "theorem isEven_add_self (n : Nat) : IsEven (n + n) := ⟨n, by omega⟩"
)

lean = Lean(lean_config(definitions))     # a core-Lean REPL session (builds once)
env = lean.base_env(definitions)          # run the preamble -> a persistent environment id

# Prove a goal that USES the definition, inside that environment:
ok = lean.check("theorem t (n : Nat) : IsEven (n + n) := isEven_add_self n", env)
print("with the definitions loaded :", ok.feedback)

# The SAME proof with no environment (definitions absent) fails — `IsEven` is unknown:
bare = lean.check("theorem t (n : Nat) : IsEven (n + n) := isEven_add_self n", None)
print("without them                :", bare.feedback.splitlines()[0])

with the definitions loaded : ✓ valid — the proof compiles with no errors and no `sorry`.
without them                : ✗ errors:


## 2. What the agent works on: a `Problem`

Everything the agent runs is a `Problem` — four fields, no magic:

- **`name`** — a label (used in the log folder name).
- **`benchmark`** — a label that groups runs.
- **`preamble`** — Lean code loaded into the environment *before* the proof, and also shown to the model in its system prompt. **This is where the definitions/lemmas go** — the knob the hypothesis turns.
- **`statement`** — the goal: a `theorem ... :=` with **no** proof (the agent supplies it).

Let's build one **by hand** and solve it. `solve` pre-loads the preamble into both Lean and the model, lets the agent iterate with `lean_check`, grades it (a real, complete proof of *this* statement), and logs the run.

In [3]:
print(definitions)

def IsEven (n : Nat) : Prop := ∃ k, n = 2 * k
theorem isEven_add_self (n : Nat) : IsEven (n + n) := ⟨n, by omega⟩


In [4]:
model = build_model()

problem = Problem(
    name="even_demo",
    benchmark="demo",
    preamble=definitions,                                   # the IsEven def + lemma from section 1
    statement="theorem target (n : Nat) : IsEven (n + n)",  # the goal — note: no `:= proof`
)

result = solve(problem, lean=lean, model=model, max_steps=5)
print(result["passed"], "in", result["steps"], "steps")
print("logged to:", result["run_dir"], "(open transcript.yaml inside)")

True in 3 steps
logged to: /Users/michaeltheologitis/Code/lean-agent/logs/20260621T143939Z-demo-even_demo (open transcript.yaml inside)


## 3. Where problems usually come from: `.lean` files

You rarely build a `Problem` by hand. Usually you have a **`.lean` file**, and the harness builds the `Problem` for you.

A `.lean` file is **Lean 4 source code** — a *different language from Python*. Python only drives the agent; the actual math lives in Lean. So in a Putnam/MiniF2F file:

- `import Mathlib` — Lean importing the Mathlib math library (this is Lean's `import`, **not** Python's).
- `open ...` — opens a Lean namespace so names are shorter.
- `theorem foo ... := sorry` — a theorem whose proof is the placeholder `sorry` (the thing to fill in).

The harness reads this text and **splits it at the last `theorem`**: everything above becomes the `preamble`, the theorem becomes the `statement`. `load(...)` does exactly that — building the same kind of `Problem` you made by hand:

In [5]:
from benchmarks import DATA, load

raw = (DATA / "minif2f" / "mathd_algebra_116.lean").read_text()
print("=== the raw .lean file (this is LEAN code, not Python) ===")
print(raw)

(p,) = load("minif2f", names=["mathd_algebra_116"])   # reads the file and splits it into a Problem
print("=== what load() made of it ===")
print("preamble (pre-loaded) :", repr(p.preamble))
print("statement (the goal)  :", repr(p.statement))
# (We don't *solve* this one here — MiniF2F needs Mathlib. Set LEAN_PROJECT to a built
#  Mathlib project, then: uv run python run.py --benchmark minif2f)

=== the raw .lean file (this is LEAN code, not Python) ===
import Mathlib

set_option maxHeartbeats 0

open BigOperators Real Nat Topology Rat

theorem mathd_algebra_116 (k x : ℝ) (h₀ : x = (13 - Real.sqrt 131) / 4)
    (h₁ : 2 * x ^ 2 - 13 * x + k = 0) : k = 19 / 4 := by sorry
=== what load() made of it ===
preamble (pre-loaded) : 'import Mathlib\n\nset_option maxHeartbeats 0\n\nopen BigOperators Real Nat Topology Rat'
statement (the goal)  : 'theorem mathd_algebra_116 (k x : ℝ) (h₀ : x = (13 - Real.sqrt 131) / 4)\n    (h₁ : 2 * x ^ 2 - 13 * x + k = 0) : k = 19 / 4'


## 4. Two ways to give the agent a new problem

Both produce the same `Problem` object that `solve` runs:

**(a) As a file (the usual way).** Write a `.lean` file into a folder under `benchmarks/data/`:
- a benchmark: `benchmarks/data/<benchmark>/<name>.lean`
- an experiment: `benchmarks/data/experiments/<name>/<condition>.lean` (one file per condition)

Put your imports + definitions, then the goal as the **last** `theorem ... := sorry`. Then `load("<benchmark>")` or `load_experiment("<name>")` reads every `.lean` in that folder and builds the Problems. For Mathlib problems, set `LEAN_PROJECT` to a built Mathlib project.

**(b) In Python (ad hoc).** Construct `Problem(name=..., benchmark=..., preamble=..., statement=...)` directly, exactly like section 2 — handy for quick experiments without touching the filesystem.

## 5. The hypothesis: same goal, with vs. without the definition

`even_self` is an experiment with two `.lean` files = two conditions of the *same* proposition: `notated.lean` gives the `IsEven` definition + lemma, `raw.lean` states the desugared `∃ k, n + n = 2 * k` with nothing given. `load_experiment` reads both files and splits each into a `Problem` (same as section 3). Run both and compare — this is the whole study, in miniature.

In [7]:
from benchmarks import load_experiment

for cond in load_experiment("even_self"):   # -> two Problem objects, one per .lean file
    r = solve(cond, lean=lean, model=model, max_steps=6)
    print(f"{cond.name:20s} passed={r['passed']}  steps={r['steps']}  out_tokens={r['output_tokens']}")

even_self/notated    passed=True  steps=3  out_tokens=60
even_self/raw        passed=True  steps=4  out_tokens=145


In [8]:
print(r)

{'problem': 'even_self/raw', 'benchmark': 'experiment', 'passed': True, 'steps': 4, 'input_tokens': 4380, 'output_tokens': 145, 'error': None, 'run_dir': '/Users/michaeltheologitis/Code/lean-agent/logs/20260621T144122Z-experiment-even_self-raw'}


## 6. Reading the logs

Every `solve` writes a folder under `logs/<run-folder>/`:
- **`run.json`** — the full structured record (every step + token usage).
- **`transcript.yaml`** — the top-down conversation lineage (system / user / assistant / tool-call / tool-response). Open this to see exactly what the model saw and every `lean_check` it tried.

The same runs from the command line: `uv run python run.py --experiment even_self`.

## 7. Using a different provider (Token Factory + DeepSeek)

Nothing here is OpenAI-specific — the agent speaks to any **OpenAI-compatible** endpoint, so you swap the provider/model just by changing env vars (resolution is OpenAI-first; see `settings.py`). Below we route through **Token Factory** (Nebius) running a **DeepSeek** model instead of `gpt-5.4-mini`. The `OPENAI_*` vars are really just "the OpenAI-compatible endpoint," so we reuse them here; there's also a dedicated `TOKEN_FACTORY_*` slot you can set in `.env`.

In [ ]:
# Swap the provider/model by changing env vars, then clear the (cached) settings and rebuild.
# Here: Token Factory (Nebius) + a DeepSeek model, via the OpenAI-compatible OPENAI_* slot.
os.environ["OPENAI_API_KEY"]  = "tf-..."                                    # your Token Factory key
os.environ["OPENAI_MODEL_ID"] = "deepseek-ai/DeepSeek-V4-Pro"               # a DeepSeek model
os.environ["OPENAI_API_BASE"] = "https://api.tokenfactory.nebius.com/v1/"   # Token Factory endpoint
get_settings.cache_clear()                                                  # settings are cached
print("provider now:", get_settings().model_id, "@", get_settings().api_base)

deepseek = build_model()
# Then run any Problem on DeepSeek instead of gpt-5.4-mini — same solve(), just a different model:
# print(solve(problem, lean=lean, model=deepseek, max_steps=6)["passed"])